In [25]:
from pathlib import Path
from src.db import get_connection
import sqlite3

import torch
import numpy as np
import pandas as pd


DATABASE_PATH = Path.home() / "HDD/Datasets/annas_archive_spotify_2025_07/spotify_clean_playlists.sqlite3"

In [2]:
conn  = get_connection(DATABASE_PATH)
print("Connected to database")

Connected to database


In [3]:
def fetch_topn_playlist_tracks(conn, n):
    """Fetch all valid (playlist_rowid, track_rowid) pairs for the top N playlists
    by followers. DISTINCT ensures each track appears at most once per playlist.
    """
    query = """
        SELECT DISTINCT pt.playlist_rowid, pt.track_rowid
        FROM playlist_tracks pt
        WHERE pt.playlist_rowid IN (
            SELECT rowid 
            FROM playlists 
            ORDER BY followers_total 
            DESC LIMIT ?
        )
          AND pt.is_episode = 0
          AND pt.is_local = 0
          AND pt.track_rowid IS NOT NULL
    """
    return pd.read_sql(query, conn, params=[n])

N = 32768
_playlist_tracks = fetch_topn_playlist_tracks(conn, N)
_playlist_tracks

,playlist_rowid,track_rowid
0,3,1
1,3,4
2,3,8
3,3,9
4,3,10
...,...,...
4343499,9882401,45162248
4343500,9882401,45162240
4343501,9882401,149605019
4343502,9882401,45162245


In [22]:
def filter_kcore(pt: pd.DataFrame, k: int, min_playlist_len: int=2) -> pd.DataFrame:
    """
    Iteratively remove tracks appearing in fewer than k playlists and playlists
    with fewer than min_playlist_len tracks, until the result is stable.

    A playlist with only one remaining track produces no center-context pairs
    and is useless for skip-gram training, hence min_playlist_len=2 by default.
    """
    while True:
        n_before = len(pt)

        track_counts = pt["track_rowid"].value_counts()
        pt = pt[pt["track_rowid"].map(track_counts) >= k]

        playlist_counts = pt["playlist_rowid"].value_counts()
        pt = pt[pt["playlist_rowid"].map(playlist_counts) >= min_playlist_len]

        if len(pt) == n_before:
            break
    return pt.reset_index(drop=True)


playlist_tracks = filter_kcore(_playlist_tracks, k=2)
print(f"Interactions    : {len(playlist_tracks):,}")
print(f"Unique tracks   : {playlist_tracks['track_rowid'].nunique():,}")
print(f"Playlists       : {playlist_tracks['playlist_rowid'].nunique():,}")
print(f"Min playlist len: {playlist_tracks["playlist_rowid"].value_counts().min()}")
print(f"Min track it.   : {playlist_tracks["track_rowid"].value_counts().min()}")

Interactions    : 3,025,875
Unique tracks   : 596,325
Playlists       : 32,292
Min playlist len: 2
Min track it.   : 2


In [23]:
def build_vocab(pt: pd.DataFrame) -> pd.DataFrame:
    """Build a track_rowid → track_id mapping from the filtered interaction table.

    Assigns a contiguous integer index to each unique track_rowid. The resulting
    DataFrame is indexed by track_rowid for fast join-based lookups.
    """
    unique_tracks = pt["track_rowid"].unique()
    return pd.DataFrame({
        "track_rowid": unique_tracks,
        "track_id": np.arange(len(unique_tracks)), 
    }).set_index("track_rowid")

vocab = build_vocab(playlist_tracks)
vocab

,track_id
track_rowid,
1,0
4,1
8,2
9,3
10,4
...,...
58565959,596320
99478108,596321
11762698,596322


In [33]:
def map_vocab(pt: pd.DataFrame, vocab: pd.DataFrame) -> pd.DataFrame:
    """Replace track_rowid with the contiguous track_id from the vocab."""
    return pt.join(vocab, on="track_rowid").drop(columns=["track_rowid"])

playlist_tracks = map_vocab(playlist_tracks, vocab)
playlist_tracks

,playlist_rowid,track_id
0,3,0
1,3,1
2,3,2
3,3,3
4,3,4
...,...,...
3025870,9882401,407568
3025871,9882401,407566
3025872,9882401,407565
3025873,9882401,596263


In [34]:
def build_weights(pt: pd.DataFrame) -> torch.Tensor:
    """Build a normalised negative-sampling weight tensor aligned to track_id order.

    Weights are proportional to freq^0.75 (word2vec unigram smoothing): frequent
    tracks are sampled more often as negatives, but the exponent dampens the
    dominance of the very top tracks.

    The returned tensor has shape (vocab_size,) where position i is the sampling
    weight for track_id=i, ready for torch.multinomial.
    """
    freq = pt["track_id"].value_counts()
    weights = torch.tensor((freq ** 0.75).sort_index().to_numpy(), dtype=torch.float32)
    return weights / weights.sum()

weights = build_weights(playlist_tracks)
weights

tensor([2.2672e-05, 2.3973e-06, 1.7616e-05,  ..., 9.3686e-07, 9.3686e-07,
        9.3686e-07])

In [ ]:
def flatten(pt: pd.DataFrame) -> dict[int, list[int]]:
    """Group interactions into a playlist → [track_ids] dictionary."""
    return pt.groupby("playlist_rowid")["track_id"].apply(list).to_dict()
